# Evaluating Expertise Representation Methods for Reviewer–Proposal Matching

This notebook demonstrates a reproducible pipeline for comparing six different
approaches to matching scientific reviewers with proposals:

| Method | Type |
|---|---|
| Keywords | Pre-computed keyword overlap |
| TF-IDF | Sparse bag-of-words |
| LDA | Latent Dirichlet Allocation (20 topics) |
| SPECTER2 | Scientific document transformer |
| SentenceTransformer | Dense sentence embeddings |
| GPT-4o-mini | LLM-based structured scoring |

## How to Run

1. Install dependencies: `pip install -r requirements.txt`
2. Generate demo data: `python -m src.dummy_data` (requires ADS API key)
3. Run this notebook cell by cell or use **Run All**.

## Inputs
- `data/demo/proposals.csv` — Proposal abstracts (held-out papers)
- `data/demo/reviewers.csv` — Reviewer names and IDs
- `data/demo/reviewer_abstracts.json` — Reviewer paper corpora
- `data/demo/ground_truth.csv` — True proposal→reviewer mapping

## Outputs
- Summary metrics table (MRR, Hit@25, Median Rank, Z-Score)
- Statistical significance tests (Wilcoxon)
- Publication-quality figures (heatmaps, KDE, boxplots, rainclouds)

## 1. Setup & Global Configuration

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd
import torch

from src import config
from src.preprocessing import clean_text
from src.data_loader import (
    load_proposals, load_reviewers, load_reviewer_abstracts,
    load_ground_truth, load_precomputed_scores,
)
from src.embeddings import get_embedder, EMBEDDER_REGISTRY
from src.metrics import (
    compute_ranking_metrics, summarise_metrics,
    pairwise_wilcoxon, compute_ndcg_per_proposal,
)
from src.reporting import report_all_methods, to_latex
from src import plotting

# Reproducibility: set all random seeds globally
np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.RANDOM_SEED)

warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

%matplotlib inline

print(f'Data directory: {config.DATA_DIR}')
print(f'Random seed:    {config.RANDOM_SEED}')

## 2. Load Data

In [ ]:
# Load all data files from the configured data directory
proposals_df = load_proposals()
reviewers_df = load_reviewers()
reviewer_abstracts = load_reviewer_abstracts()
ground_truth = load_ground_truth()

print(f'Proposals:  {len(proposals_df)}')
print(f'Reviewers:  {len(reviewers_df)}')
print(f'Ground truth pairs: {len(ground_truth)}')

proposals_df.head()

## 3. Prepare Text Corpora

Combine each reviewer's paper abstracts into a single document.
Proposal texts come directly from the abstracts.

In [ ]:
# Build parallel lists matching reviewers_df order
reviewer_ids = reviewers_df['reviewer_id'].astype(str).tolist()
reviewer_names = reviewers_df['full_name'].tolist()

# Concatenate all abstracts per reviewer into a single document
reviewer_texts = []
for name in reviewer_names:
    data = reviewer_abstracts.get(name, {})
    abstracts = data.get('abstracts', [])
    reviewer_texts.append(' '.join(abstracts) if abstracts else '')

# Proposal texts: title + abstract
proposal_ids = proposals_df['proposal_id'].astype(str).tolist()
proposal_texts = (
    proposals_df['title'].fillna('') + ' ' + proposals_df['abstract'].fillna('')
).tolist()

print(f'Reviewer texts prepared: {len(reviewer_texts)}')
print(f'Proposal texts prepared: {len(proposal_texts)}')

## 4. Generate Embeddings & Compute Scores

Run each embedding method and collect the resulting score matrices.
Keywords and GPT-4o-mini load pre-computed CSVs; the rest compute on-the-fly.

In [ ]:
import time

# Select which methods to run
# For a quick demo, you can comment out slow methods (specter, sentence_transformer)
methods_to_run = [
    'keywords',
    'tfidf',
    'lda',
    # 'specter',              # Uncomment if SPECTER2 + adapters are installed
    # 'sentence_transformer', # Uncomment for SentenceTransformer
    'gpt4o',
]

score_matrices = {}  # {method_name: pd.DataFrame}
timings = {}         # {method_name: seconds}

for method_key in methods_to_run:
    embedder = get_embedder(method_key)
    print(f'\n{"="*60}')
    print(f'Running: {embedder.name}')
    print(f'{"="*60}')

    t0 = time.time()
    scores = embedder.compute_scores(
        proposal_texts, reviewer_texts, proposal_ids, reviewer_ids
    )
    elapsed = time.time() - t0

    score_matrices[embedder.name] = scores
    timings[embedder.name] = elapsed
    print(f'  Shape: {scores.shape}  |  Time: {elapsed:.2f}s')

print(f'\nCompleted {len(score_matrices)} methods.')
pd.Series(timings, name='seconds').to_frame().T

## 5. Compute Ranking Metrics

For each method, compute per-proposal MRR, Rank, Hit@25, and Z-Score.
Then summarise across all proposals.

In [ ]:
# Compute per-proposal metrics for each method
all_metrics = {}  # {method_name: {proposal_id: {metric: value}}}

for method_name, scores_df in score_matrices.items():
    m = compute_ranking_metrics(scores_df, ground_truth)
    all_metrics[method_name] = m
    print(f'{method_name}: {len(m)} proposals evaluated')

# Summary table
summary = report_all_methods(all_metrics)
summary

## 6. Statistical Significance (Wilcoxon)

Paired Wilcoxon signed-rank test comparing each method against the
Keywords baseline on MRR.

In [ ]:
# Pairwise Wilcoxon vs Keywords baseline
if 'Keywords' in all_metrics:
    wilcoxon_results = pairwise_wilcoxon(
        all_metrics, baseline_name='Keywords', metric_key='mrr'
    )
    display(wilcoxon_results)
else:
    print('Keywords method not available — skipping significance tests.')

## 7. Visualisations

In [ ]:
# Heatmaps: score matrices side by side
plotting.plot_heatmaps(
    score_dfs=list(score_matrices.values()),
    titles=list(score_matrices.keys()),
    save_path=config.OUTPUT_DIR / 'heatmaps_allmethods.png',
)

In [ ]:
# Score distributions (KDE)
plotting.plot_score_distributions(
    score_dfs=list(score_matrices.values()),
    labels=list(score_matrices.keys()),
    save_path=config.OUTPUT_DIR / 'distribution_allmethods.png',
)

In [ ]:
# Rank boxplot
plotting.plot_rank_boxplot(
    method_metrics=all_metrics,
    save_path=config.OUTPUT_DIR / 'rank_boxplot.png',
)

## 8. LaTeX Table Export

In [ ]:
# Export summary table as LaTeX for the manuscript
latex_str = to_latex(summary)
print(latex_str)

# Save to file
tex_path = config.OUTPUT_DIR / 'metrics_table.tex'
with open(tex_path, 'w') as f:
    f.write(latex_str)
print(f'\nSaved to {tex_path}')

## 9. Discussion

The results above show how different expertise representation methods
compare in their ability to rank the true expert reviewer highly.

**Key observations**:
- Methods producing higher MRR and lower Median Rank are better at
  surfacing the correct reviewer.
- Statistical significance (Wilcoxon p-values) indicates whether
  differences vs. the Keywords baseline are robust.
- The score distribution plots reveal how discriminative each method is
  (wider separation between Expert and Non-Expert scores is desirable).

---
*Pipeline code: `src/` | Data: `data/demo/` | Figures: `output/`*